In [1]:
pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.8 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

from catboost import CatBoostClassifier, Pool

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "ml_dataset_expanded_forecastsafe.csv"   # your dataset file
OUT_DIR = "artifacts/model"
os.makedirs(OUT_DIR, exist_ok=True)

RISK_MODEL_OUT = os.path.join(OUT_DIR, "risk_model_v4.cbm")
META_OUT = os.path.join(OUT_DIR, "model_meta_v4.json")

RANDOM_STATE = 42

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

# -----------------------------
# FEATURES (must match FastAPI)
# -----------------------------
CATEGORICAL = [
    "region",
    "location",
    "season",
    "lag_top_1",
    "lag_top_2",
    "lag_top_3",
]

NUMERIC = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_cases_2", "lag_cases_3",
    "lag_has_1", "lag_has_2", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6",
]

LABEL = "has_offence"

required = CATEGORICAL + NUMERIC + [LABEL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

# -----------------------------
# CLEAN TYPES
# -----------------------------
for c in CATEGORICAL:
    df[c] = df[c].fillna("None").astype(str)

for c in NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(float)

df[LABEL] = pd.to_numeric(df[LABEL], errors="coerce").fillna(0).astype(int).clip(0, 1)

# Optional sanity filter
df = df[(df["month_num"] >= 1) & (df["month_num"] <= 12)].copy()

X = df[CATEGORICAL + NUMERIC]
y = df[LABEL]

# CatBoost needs indices of categorical columns (by position)
cat_feature_indices = list(range(len(CATEGORICAL)))  # first 6 columns are categorical

# -----------------------------
# SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_indices)

# -----------------------------
# TRAIN CatBoost (Risk)
# -----------------------------
# This setting usually works well for tabular + mixed categorical/numeric
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    verbose=100,
    auto_class_weights="Balanced",   # important for imbalanced offences
    early_stopping_rounds=100
)

model.fit(train_pool, eval_set=test_pool)

# -----------------------------
# EVALUATE
# -----------------------------
proba = model.predict_proba(test_pool)[:, 1]
pred = (proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba)
ap = average_precision_score(y_test, proba)

print("\n=== RISK MODEL (CatBoost) RESULTS ===")
print("ROC-AUC:", round(auc, 4))
print("PR-AUC :", round(ap, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("Classification Report:\n", classification_report(y_test, pred, digits=4))

# -----------------------------
# SAVE MODEL (CBM is safest)
# -----------------------------
model.save_model(RISK_MODEL_OUT)
print("\nSaved:", RISK_MODEL_OUT)

# -----------------------------
# SAVE META (for FastAPI)
# -----------------------------
meta = {
    "version": "v4",
    "risk_model": {
        "type": "catboost",
        "file": "risk_model_v4.cbm",
        "label": LABEL
    },
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "notes": {
        "forecast_safe": True,
        "rolling_features": ["roll_3_cases", "roll_6_cases"],
        "trend_features": ["trend_3", "trend_6"]
    }
}

with open(META_OUT, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", META_OUT)


0:	test: 0.8841935	best: 0.8841935 (0)	total: 119ms	remaining: 3m 58s
100:	test: 0.9718122	best: 0.9718122 (100)	total: 4.95s	remaining: 1m 33s
200:	test: 0.9809615	best: 0.9809615 (200)	total: 11.1s	remaining: 1m 39s
300:	test: 0.9841926	best: 0.9841926 (300)	total: 16.7s	remaining: 1m 34s
400:	test: 0.9860874	best: 0.9860874 (400)	total: 23.3s	remaining: 1m 33s
500:	test: 0.9870224	best: 0.9870853 (498)	total: 28.7s	remaining: 1m 25s
600:	test: 0.9873148	best: 0.9873148 (600)	total: 35.9s	remaining: 1m 23s
700:	test: 0.9876911	best: 0.9877154 (698)	total: 41.1s	remaining: 1m 16s
800:	test: 0.9882622	best: 0.9882832 (799)	total: 47.9s	remaining: 1m 11s
900:	test: 0.9887585	best: 0.9888693 (894)	total: 53s	remaining: 1m 4s
1000:	test: 0.9889690	best: 0.9890378 (954)	total: 59.4s	remaining: 59.2s
1100:	test: 0.9892083	best: 0.9892096 (1099)	total: 1m 5s	remaining: 53.9s
1200:	test: 0.9893099	best: 0.9893480 (1183)	total: 1m 12s	remaining: 48.1s
1300:	test: 0.9895040	best: 0.9895040 (130

In [4]:
import os
import json
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
DATA_PATH = "ml_dataset_expanded_forecastsafe.csv"
OUT_DIR = "artifacts/model"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_OUT = os.path.join(OUT_DIR, "risk_model_v5_timeaware.cbm")
META_OUT = os.path.join(OUT_DIR, "model_meta_v5.json")

TRAIN_END_YEAR = 2022      # past
TEST_START_YEAR = 2023     # future

RANDOM_STATE = 42

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
df = pd.read_csv(DATA_PATH)

# -------------------------------------------------
# FEATURES (must match FastAPI exactly)
# -------------------------------------------------
CATEGORICAL = [
    "region",
    "location",
    "season",
    "lag_top_1",
    "lag_top_2",
    "lag_top_3",
]

NUMERIC = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_cases_2", "lag_cases_3",
    "lag_has_1", "lag_has_2", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6",
]

LABEL = "has_offence"

# -------------------------------------------------
# BASIC CLEANING
# -------------------------------------------------
for c in CATEGORICAL:
    df[c] = df[c].fillna("None").astype(str)

for c in NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(float)

df[LABEL] = pd.to_numeric(df[LABEL], errors="coerce").fillna(0).astype(int).clip(0, 1)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

# -------------------------------------------------
# TIME-BASED SPLIT
# -------------------------------------------------
train_df = df[df["year"] <= TRAIN_END_YEAR].copy()
test_df = df[df["year"] >= TEST_START_YEAR].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split is empty. Check year ranges.")

X_train = train_df[CATEGORICAL + NUMERIC]
y_train = train_df[LABEL]

X_test = test_df[CATEGORICAL + NUMERIC]
y_test = test_df[LABEL]

cat_feature_indices = list(range(len(CATEGORICAL)))

train_pool = Pool(
    X_train, y_train,
    cat_features=cat_feature_indices
)

test_pool = Pool(
    X_test, y_test,
    cat_features=cat_feature_indices
)

# -------------------------------------------------
# TRAIN CatBoost (Risk Model)
# -------------------------------------------------
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    auto_class_weights="Balanced",
    early_stopping_rounds=200,
    verbose=200,
)

model.fit(train_pool, eval_set=test_pool)

# -------------------------------------------------
# EVALUATION (REAL-WORLD)
# -------------------------------------------------
proba = model.predict_proba(test_pool)[:, 1]
pred = (proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)

print("\n=== TIME-BASED RISK MODEL EVALUATION ===")
print("Train years ≤", TRAIN_END_YEAR)
print("Test years ≥", TEST_START_YEAR)
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC :", round(pr_auc, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

# -------------------------------------------------
# SAVE MODEL
# -------------------------------------------------
model.save_model(MODEL_OUT)

# -------------------------------------------------
# SAVE META
# -------------------------------------------------
meta = {
    "version": "v5",
    "risk_model": {
        "type": "catboost",
        "file": "risk_model_v5_timeaware.cbm",
        "label": LABEL,
        "train_years": f"<= {TRAIN_END_YEAR}",
        "test_years": f">= {TEST_START_YEAR}",
    },
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "evaluation": {
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "threshold": 0.5,
    },
    "forecast_safe": True,
}

with open(META_OUT, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved model:", MODEL_OUT)
print("Saved meta :", META_OUT)


0:	test: 0.8201812	best: 0.8201812 (0)	total: 42.3ms	remaining: 2m 6s
200:	test: 0.8033581	best: 0.8727336 (10)	total: 6.39s	remaining: 1m 29s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.8727336227
bestIteration = 10

Shrink model to first 11 iterations.

=== TIME-BASED RISK MODEL EVALUATION ===
Train years ≤ 2022
Test years ≥ 2023
ROC-AUC: 0.8727
PR-AUC : 0.7051

Confusion Matrix:
[[4300 2115]
 [ 194 2753]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9568    0.6703    0.7883      6415
           1     0.5655    0.9342    0.7045      2947

    accuracy                         0.7534      9362
   macro avg     0.7612    0.8022    0.7464      9362
weighted avg     0.8337    0.7534    0.7620      9362


Saved model: artifacts/model/risk_model_v5_timeaware.cbm
Saved meta : artifacts/model/model_meta_v5.json
